In [1]:
import sys

from src.data.load_cifar100 import get_cifar100_loaders, create_and_load_subset

sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import numpy as np
import pandas as pd
import os
import datetime
from src.models.masked_autoencoder import MAE
from src.data.load_cifar10 import get_cifar10_loaders
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [2]:
# Dane
# train_loader, val_loader, test_loader = get_cifar10_loaders(batch_size=64)
# train_loader, val_loader, test_loader = get_cifar100_loaders(batch_size=64)
_, selected_classes, train_loader, val_loader, test_loader = create_and_load_subset(
    num_classes=2, 
    batch_size=64
)
print(f"Trenowanie na klasach: {selected_classes}")

Wylosowano nowe klasy: [34, 43]
Trenowanie na klasach: [34, 43]


In [3]:
# Model
model = MAE().to(device)
# criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [4]:
# Trening
BASE_DIR = os.getcwd()
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar10')
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar100')
print(save_dir)
# writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_mae', 'cifar10')
writer_dir = os.path.join(BASE_DIR, '..', 'training_results', 'tb_mae', 'cifar100')
os.makedirs(save_dir, exist_ok=True)
os.makedirs(writer_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(os.path.join(writer_dir, f'mae_{timestamp}'))

num_epochs = 250

train_losses = []
val_losses = []
epoch_number = 0
best_val_loss = float('inf')
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    
    for batch_idx, (image, _) in enumerate(train_loader):
        image = image.to(device)
        
        reconstructed, x_masked, mask = model(image)
        loss = model.compute_loss(image, reconstructed, mask)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f"  [{epoch+1}/{num_epochs}] Batch {batch_idx}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f}")
            
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    writer.add_scalar('Loss/train', train_loss, epoch)

    # validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for image, _ in val_loader:
            image = image.to(device)

            reconstructed, x_masked, mask = model(image)
            loss = model.compute_loss(image, reconstructed, mask)
            val_loss += loss.item()
            
        if epoch % 5 == 0:  
            n_images = min(8, image.size(0))
            
            # Oryginalne obrazy
            writer.add_images('Original', image[:n_images], epoch)
            
            # Zamaskowane obrazy
            writer.add_images('Masked', x_masked[:n_images], epoch)
            
            # Rekonstrukcje
            writer.add_images('Reconstructed', reconstructed[:n_images], epoch)
            
            # Wizualizacja maski (powtórzona dla 3 kanałów)
            mask_vis = mask[:n_images].repeat(1, 3, 1, 1)
            writer.add_images('Mask', mask_vis, epoch)
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    writer.add_scalar('Loss/val', val_loss, epoch)
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")


    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'latent_dim': 256,
            'train_losses': train_losses,
            'val_losses': val_losses
        }
        timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        # checkpoint_path = os.path.join(save_dir, f'masked_autoencoder_cifar10_best_{timestamp}.pt')
        checkpoint_path = os.path.join(save_dir, f'masked_autoencoder_cifar100_best_{timestamp}.pt')
        torch.save(checkpoint, checkpoint_path)


df = pd.DataFrame({
    'epoch': range(1, num_epochs + 1),
    'train_loss': train_losses,
    'val_loss': val_losses
})
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# history_csv = os.path.join(save_dir, f'masked_autoencoder_cifar10_training_results_{timestamp}.csv')
history_csv = os.path.join(save_dir, f'masked_autoencoder_cifar100_training_results_{timestamp}.csv')
df.to_csv(history_csv, index=False)


C:\Users\martu\Desktop\studia\magisterka\2sem\ml_projekt\src\notebooks_test_train\..\training_results\masked_autoencoder\cifar100
  [1/250] Batch 0/13 Loss: 0.2130
Epoch [1/250]
Train Loss: 0.1264
Val Loss:   0.1885
  [2/250] Batch 0/13 Loss: 0.1026
Epoch [2/250]
Train Loss: 0.1005
Val Loss:   0.2574
  [3/250] Batch 0/13 Loss: 0.0987
Epoch [3/250]
Train Loss: 0.0925
Val Loss:   0.1861
  [4/250] Batch 0/13 Loss: 0.0982
Epoch [4/250]
Train Loss: 0.0893
Val Loss:   0.1242
  [5/250] Batch 0/13 Loss: 0.0880
Epoch [5/250]
Train Loss: 0.0845
Val Loss:   0.0989
  [6/250] Batch 0/13 Loss: 0.0791
Epoch [6/250]
Train Loss: 0.0831
Val Loss:   0.0860
  [7/250] Batch 0/13 Loss: 0.0828
Epoch [7/250]
Train Loss: 0.0812
Val Loss:   0.0933
  [8/250] Batch 0/13 Loss: 0.0746
Epoch [8/250]
Train Loss: 0.0807
Val Loss:   0.0910
  [9/250] Batch 0/13 Loss: 0.0838
Epoch [9/250]
Train Loss: 0.0790
Val Loss:   0.1028
  [10/250] Batch 0/13 Loss: 0.0860
Epoch [10/250]
Train Loss: 0.0771
Val Loss:   0.0809
  [11/25

In [8]:
# save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar10')
# best_checkpoint = torch.load(os.path.join(save_dir, 'masked_autoencoder_cifar10_best_20260107_192113.pt'))
# 
save_dir = os.path.join(BASE_DIR, '..', 'training_results', 'masked_autoencoder', 'cifar100')
best_checkpoint = torch.load(os.path.join(save_dir, 'masked_autoencoder_cifar100_best_20260112_113108.pt'))
model.load_state_dict(best_checkpoint['model_state_dict'])
correct_test = 0
total_test = 0
test_loss = 0

model.eval()
print("\nCalculating metrics on test set...")
with torch.no_grad():
    for image, _ in test_loader:
        image = image.to(device)

        reconstructed, x_masked, mask = model(image)
        loss = model.compute_loss(image, reconstructed, mask)
        test_loss += loss.item()
      
test_loss /= len(test_loader)
writer.add_scalar('Loss/test', test_loss)
print("FINAL EVALUATION RESULTS")
print(f"Test Loss:           {test_loss:.6f}")      
        
    


Calculating metrics on test set...
FINAL EVALUATION RESULTS
Test Loss:           0.060510
